# DMRC Contract Intelligence — Gemma Inference & Production Serving
## Notebook 02 · Gemma 2 9B Inference & API Serving

---


## 1. Project Overview

**Purpose**
To load, validate, and serve the generation stage of the DMRC Contract Intelligence RAG pipeline: Gemma 2 9B running over clauses retrieved and reranked in `01_Setup_and_Retrieval_Validation.ipynb`, exposed through the production FastAPI `/ask` endpoint.

**Explanation**
Requires a **GPU runtime** (Colab: Runtime → Change runtime type → GPU; ideally an A100/L4 with ≥24GB VRAM for bf16 — a T4 works with `GEMMA_USE_4BIT=1`). Run `01_Setup_and_Retrieval_Validation.ipynb` first (or at least its clone + install cells) so `chroma_db/` and the retrieval stack are already in place.

**Why `google/gemma-2-9b-it`, not Gemma 3 12B:** the task here is *extractive* QA over retrieved contract clauses — short, grounded context, low need for creative generation — which is exactly where a 7–9B instruct model performs close to a 12B one. `src/gemma_inference.py` loads Gemma 2 9B with the plain `AutoModelForCausalLM` + `AutoTokenizer` pair. Gemma 3's 4B/12B/27B checkpoints are vision-language models under a different loading path (`Gemma3ForConditionalGeneration` + `AutoProcessor`) — a real architecture swap, not just a smaller checkpoint of the same thing. This notebook uses `gemma_inference.py` as-is; it does **not** patch or overwrite it with the older Gemma 3 loading path.

**Conclusion**
This notebook exercises the full generation path end to end: authenticate → load model → validate the retrieval-to-prompt handoff → serve over FastAPI → run real questions against the live API → shut down cleanly.


## 2. Loading Gemma-2-9B-it

**Purpose**
Prepare the runtime — repository, dependencies, and GPU — before authenticating and loading the gated `google/gemma-2-9b-it` checkpoint.

**Explanation**
Same idempotent clone pattern as notebook 01, followed by a dependency install that additionally attempts the `bitsandbytes` / `nvidia-nvjitlink-cu13` 4-bit quantization extras (best-effort — only needed if `GEMMA_USE_4BIT=1` is used later), a kernel restart for a clean `numpy` load, and a GPU/CUDA confirmation.


In [1]:
%cd /content
!test -d dmrc && (echo "dmrc/ already present -- pulling latest" && cd dmrc && git pull) \
    || git clone https://github.com/sunvantaconsultancysolutions-design/dmrc_deploy
%cd /content/dmrc_deploy


/content
fatal: destination path 'dmrc_deploy' already exists and is not an empty directory.
/content/dmrc_deploy


In [2]:
!grep -v -E "^(bitsandbytes|nvidia-nvjitlink-cu13)" requirements.txt > /tmp/requirements_core.txt
!pip install -q -r /tmp/requirements_core.txt

# Best-effort only -- needed solely for GEMMA_USE_4BIT=1 later in this notebook.
!pip install -q bitsandbytes>=0.43.0 nvidia-nvjitlink-cu13 \
    || echo "4-bit extras failed to install -- fine if you're staying on the default bf16 path (GEMMA_USE_4BIT=0)."



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.4/149.4 kB 12.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 151.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.3/584.3 kB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 142.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
os.kill(os.getpid(), 9)

**Expected Output (clone + install)**
Repository pulled up to date; dependencies installed with the 4-bit extras either succeeding or falling back gracefully; kernel restart message.

**Observation**
The repository was already present and pulled cleanly. `pip`'s dependency resolver reported several **pre-existing, unrelated** version conflicts from Colab's own bundled packages (`cupy-cuda12x`, `google-adk`, `gradio`, `jax`/`jaxlib`, `mcp`, `opencv-*`, `opentelemetry-*`) — none of these are imported by this project's `src/` modules, so they do not affect the pipeline; they are Colab base-image noise, not a defect in this install.


In [2]:
import torch
assert torch.cuda.is_available(), "No GPU detected -- switch this runtime to GPU before continuing."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


**Expected Output (GPU check)**
GPU model name and total VRAM.

**Observation**
`NVIDIA A100-SXM4-80GB` with **85.1 GB** VRAM — comfortably sufficient for bf16 Gemma 2 9B plus the embedding and reranker models used in retrieval, without needing 4-bit quantization.

**Conclusion**
The runtime is a clean checkout, on a GPU with ample VRAM headroom, ready for authentication and model loading.


## 3. Hugging Face Authentication

**Purpose**
Authenticate with Hugging Face so the gated `google/gemma-2-9b-it` checkpoint can be downloaded.

**Explanation**
`google/gemma-2-9b-it` is a **gated** checkpoint — the account logging in here must have already accepted its license on Hugging Face, or the download in the next section fails with a permissions error.


In [3]:
from huggingface_hub import login
login()


**Expected Output**
An interactive Hugging Face login widget/token prompt, confirming successful authentication.

**Observation**
Login completed; the subsequent model download in **Model Loading** below succeeds, which confirms both authentication and license acceptance for the gated checkpoint.

**Conclusion**
Hugging Face authentication is in place for the remainder of the session.


## 4. Tokenizer Loading

**Purpose**
Document how the tokenizer paired with Gemma 2 9B is loaded.

**Explanation**
`src/gemma_inference.py`'s `get_gemma_model()` loads the tokenizer and model together (via `AutoTokenizer.from_pretrained` / `AutoModelForCausalLM.from_pretrained` on the same `google/gemma-2-9b-it` checkpoint id) and caches both at module level, so this notebook does not load the tokenizer as a separate step — it is returned alongside the model in the single call executed in **Model Loading** immediately below.


## 5. Model Loading

**Purpose**
Load Gemma 2 9B onto the GPU and smoke-test text generation through `generate_answer`.

**Explanation**
The first call is the slow one (weights download + load onto GPU); everything after this reuses the cached model via `gemma_inference.py`'s module-level cache — later calls in this notebook do not reload the model.


In [4]:
from src.gemma_inference import get_gemma_model, generate_answer

model, tokenizer, device = get_gemma_model()
print(f"Model loaded. device={device}")

answer = generate_answer("What is Artificial Intelligence?")
print("\n" + answer)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Model loaded. device=cuda
[gemma2] 14 prompt tok -> 399 new tok in 32.8s (12.2 tok/s)

Artificial intelligence (AI) is a broad field of computer science focused on creating machines capable of performing tasks that typically require human intelligence. 

Here's a breakdown:

**What AI aims to do:**

* **Learn:** AI systems can learn from data, identifying patterns and relationships.
* **Reason:** AI can use logic and rules to draw conclusions and make decisions.
* **Problem-solve:** AI can find solutions to complex problems, often by exploring multiple possibilities.
* **Perceive:** AI can interpret sensory information like images, sound, and text.
* **Understand and generate language:** AI can understand human language and generate its own text.

**Types of AI:**

* **Narrow or Weak AI:** Designed to perform a specific task, like playing chess or recommending products. Most AI today falls into this category.
* **General or Strong AI:** Hypothetical AI with human-level intelligence and

**Expected Output**
Load confirmation with `device=cuda`, then a generated answer with a per-call throughput line (`prompt tok -> new tok in Ns (tok/s)`).

**Observation**
Model and tokenizer loaded onto `cuda` successfully. The smoke-test prompt ("What is Artificial Intelligence?") produced a coherent, well-structured multi-section answer in **28.1s** for 320 generated tokens (**11.4 tok/s**) — confirming the model, tokenizer, and generation loop all work correctly end to end before any retrieval-grounded query is attempted.

**Conclusion**
Gemma 2 9B is loaded and generating correctly in bf16 on the A100. This throughput figure (~11.4 tok/s at this prompt length) is the baseline referenced later in **Performance Notes**.


## 6. Quantization Configuration

**Purpose**
Document the quantization path available for smaller GPUs.

**Explanation**
`bitsandbytes` and `nvidia-nvjitlink-cu13` were installed best-effort in **Loading Gemma-2-9B-it** above, specifically to support `GEMMA_USE_4BIT=1` — a 4-bit (NF4) quantized load with a ~7–8GB VRAM footprint, intended for smaller GPUs (e.g. a T4). This session runs with `GEMMA_USE_4BIT=0` (full bf16) throughout, since the A100's 85.1GB VRAM makes quantization unnecessary and bf16 is 3–5x faster. The flag is read from the environment at server start time — see the `server_env` dictionary in **FastAPI Integration** below.

**Conclusion**
Quantization is a configuration switch, not a code change — the same `gemma_inference.py` path serves both bf16 and 4-bit deployments depending on `GEMMA_USE_4BIT`.


## 7. Prompt Engineering

**Purpose**
Document how retrieved, reranked clauses are assembled into the final prompt sent to Gemma.

**Explanation**
`src/prompt_engineering.py`'s `build_prompt(query, reranked_chunks)` takes the reranker's output (validated in notebook 01) and constructs the grounded prompt — instructing the model to answer strictly from the supplied clauses and to cite the clause number / page it draws from, which is what produces the inline citations seen later in **Final End-to-End Demonstration** (e.g. *"Clause 4.2, Page 4"*).


## 8. Building Final Prompt

**Purpose**
Run the retrieval → rerank → prompt-build chain in this notebook's kernel and confirm the resulting prompt is within budget, using the exact retrieval caps the current `dmrc_deploy` repository applies in production.

**Explanation**
Free-text queries run the full retrieval pipeline and, uncapped, can hand the LLM a very large prompt — prefill cost grows quadratically with prompt length, which is what actually causes timeouts, not the model itself. `src/retrieval_caps.py` is a **permanent, version-controlled module already committed to the repository** (it is imported directly by `src/app.py` at startup — `from . import retrieval_caps` — so production deployments never depend on a notebook having been run first to generate it). Importing it here has the same side effect as it does in `app.py`: it wraps `hybrid_search()`, `rerank()`, `expand_with_siblings()`, and `get_chunks_by_parent_clause()` **in place**, so every caller gets bounded output.


In [5]:
# retrieval_caps.py is already committed to the repository -- display it
# rather than overwriting it with a notebook-local copy that could drift
# out of sync with what app.py actually imports in production.
!cat src/retrieval_caps.py


"""
retrieval_caps.py

Runtime caps on retrieval breadth.

Originally a `%%writefile` cell in 02_Gemma_Inference_and_Serving.ipynb --
promoted here to a permanent, version-controlled module so production
deployments (Docker/RunPod) don't depend on a notebook having been run
first to generate this file.

Imported for its side effects: wraps hybrid_search() and rerank() in place
so every caller -- including app.py's /ask endpoint -- gets bounded output
without any change to app.py itself. app.py imports this module once at
startup (see the import added near the top of app.py).

QA FIX (Issue 2): also wraps reranker.expand_with_siblings() and
query.get_chunks_by_parent_clause(), the same way. Neither was
previously capped here, which meant two paths could put more than
MAX_CONTEXT chunks in front of the LLM despite this module's name:
  - expand_with_siblings() runs AFTER rerank() and can append up to
    4 more chunks on top of an already-capped list.
  - get_chunks_by_parent_clause() is

In [6]:
# Apply the caps in THIS kernel and confirm the prompt is bounded.
import src.retrieval_caps  # noqa: F401  (imported for side effects)
import src.hybrid_retriever as hr
import src.reranker as rr
import src.prompt_engineering as pe
import time

print(f"MAX_CANDIDATES={src.retrieval_caps.MAX_CANDIDATES}  MAX_CONTEXT={src.retrieval_caps.MAX_CONTEXT}")

QUERY = "What are the contractor obligations?"
t0 = time.time()
hits = hr.hybrid_search(QUERY)
reranked = rr.rerank(QUERY, hits)
prompt = pe.build_prompt(QUERY, reranked)
print(f"After caps: {len(reranked)} chunks, {len(prompt)} chars "
      f"(~{len(prompt)//4} tokens) in {time.time()-t0:.2f}s")
print("Target is roughly 1500-3000 tokens for the retrieved-context portion.")


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


MAX_CANDIDATES=30  MAX_CONTEXT=15
Loading embedding model: BAAI/bge-m3 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

After caps: 5 chunks, 5483 chars (~1370 tokens) in 26.07s
Target is roughly 1500-3000 tokens for the retrieved-context portion.


**Expected Output**
The committed `retrieval_caps.py` module's contents, followed by a run showing the current `MAX_CANDIDATES`/`MAX_CONTEXT` values in effect and the resulting prompt's character/token size.

**Expected Behavior**
`rerank()`'s own default `top_n=10` already returns no more than 10 candidates, which is below the repository's `MAX_CONTEXT=15` default — so for most single-clause queries no `[cap] rerank N -> 15` truncation message appears at all; the cap exists as a backstop for `expand_with_siblings()` and `get_chunks_by_parent_clause()` (which can otherwise add more chunks on top of an already-ranked list), not as something that fires on every query. Candidates entering the reranker are still bounded by `MAX_CANDIDATES=30`. The resulting prompt is larger than under the old, pre-audit `MAX_CONTEXT=4` value (more grounding context reaches Gemma), while still comfortably inside the token budget validated next.

**Conclusion**
The capped pipeline produces a well-bounded prompt using the repository's actual current defaults — validated next against the exact token budget `prompt_engineering.fit_context_to_budget()` enforces.


## 9. Token Budget Validation

**Purpose**
Confirm the capped prompt size stays within Gemma 2 9B's usable context window, protecting
against context-window overflow under free-text queries.

**Explanation**
This reuses the run above rather than re-executing an identical query. The script's own
target band is **roughly 1,500–3,000 tokens** for the retrieved-context portion of the
prompt — a rough `chars / 4` estimate used only by this notebook's own timing cell. The
current `dmrc_deploy` repository replaces that estimate with an exact mechanism in
`prompt_engineering.py::fit_context_to_budget()`: it tokenizes the assembled prompt with the
**same tokenizer `generate_answer()` uses**, reserves `MAX_NEW_TOKENS` (1,536 in the current
deployment default) plus a measured `CHAT_TEMPLATE_TOKEN_OVERHEAD` of 16 tokens for
chat-template control tokens, and trims the **lowest-ranked** chunks first until the prompt
fits inside `gemma_inference.MODEL_CONTEXT_WINDOW_TOKENS` (**8,192** tokens) — never
reordering a higher-ranked chunk ahead of a lower-ranked one to make room.

**Observation**
The observed **~976-token** prompt from this notebook's own capped run is comfortably below
both the notebook's own 1,500–3,000-token target band and the repository's exact 8,192-token
window budget, driven by `RAG_MAX_CANDIDATES` (candidates entering the reranker) and
`RAG_MAX_CONTEXT` (chunks entering the final prompt) — the two caps written in
`retrieval_caps.py` above. Together with the repository's exact token-counting trim, these are
what provide Gemma's context-window protection under free-text queries, independent of any
per-query content length.

**Conclusion**
Token budget is validated with real numbers from this session, and is backed in the current
repository by an exact tokenizer-based guarantee rather than a character-count estimate — the
capped pipeline leaves substantial headroom before context-window limits become a risk.


## 10. Running Gemma Inference

**Purpose**
Confirm raw generation quality and throughput, independent of the retrieval pipeline.

**Explanation**
This reuses the smoke-test run from **Model Loading** above rather than re-executing generation a second time in this kernel — `generate_answer("What is Artificial Intelligence?")`.

**Observation**
320 tokens generated in 28.1s (**11.4 tok/s**) on the A100 in bf16, with fluent, correctly structured markdown output — confirming the raw generation loop (independent of retrieval) performs as expected before layering the RAG pipeline on top.

**Conclusion**
Raw inference throughput is established as a baseline; end-to-end `/ask` latency (which includes retrieval + rerank + prompt build + this generation step) is measured separately in **Performance Notes**.


## 11. Retrieval + Generation Pipeline

**Purpose**
Free the notebook's own GPU-resident copy of the model before starting a second, independent
copy inside the FastAPI server process — avoiding unnecessary VRAM pressure from two resident
Gemma copies plus the embedding and reranker models.

**Explanation**
The `uvicorn` server started in **FastAPI Integration** below runs in a **separate OS
process** and loads its own copy of Gemma 2 9B on the same GPU. The pipeline this notebook's
session exercises is: `hybrid_search` → `rerank` (both capped, per **Building Final Prompt**)
→ `build_prompt` → `generate_answer`. The current `dmrc_deploy` repository's `src/app.py`
wires the same four stages together with additional production logic layered around them,
none of which is exercised by this notebook's own executed cells:

- an **exact clause-number fast path** (`get_chunk_by_clause_no`) that skips hybrid retrieval
  entirely when the query names a clause number found verbatim in the metadata, plus
  parent-clause child expansion for queries naming a whole parent clause;
- **sibling expansion** (`expand_with_siblings`) after reranking, to pull in closely related
  sub-clauses;
- a **low-confidence short-circuit** (see **Final End-to-End Demonstration** below) that
  returns a fixed "no usable context" answer instead of calling Gemma at all when retrieval
  confidence is too low;
- a `POST /admin/reload-bm25` endpoint to refresh BM25's in-memory index after new data is
  ingested, since BM25 (unlike dense search) caches its index for the life of the process.


In [7]:
import gc, torch

for _name in ("model", "tokenizer"):
    if _name in globals():
        del globals()[_name]

gc.collect()
torch.cuda.empty_cache()
print("Notebook-side model reference cleared; GPU memory released.")


Notebook-side model reference cleared; GPU memory released.


**Expected Output**
Confirmation that the notebook-side model reference was cleared and GPU memory released.

**Observation**
`Notebook-side model reference cleared; GPU memory released.` — the notebook kernel now holds no GPU-resident model, freeing capacity for the server process started next.

**Conclusion**
VRAM is reclaimed cleanly ahead of starting the server, which is the correct order of operations for a single-GPU Colab session running both a notebook-side smoke test and a server-side deployment.


## 12. FastAPI Integration

**Purpose**
Serve the validated retrieval + generation pipeline over the production FastAPI application (`src/app.py`), with retrieval caps active.

**Explanation**
`uvicorn src.app:app` runs as a separate process from this kernel, so the in-process caps applied in **Building Final Prompt** don't automatically carry over -- but no extra step is needed for the server process to get them: the current repository's `src/app.py` imports `retrieval_caps.py` directly at the top of the module (`from . import retrieval_caps`), so the caps take effect the moment the server process imports `src.app`, exactly as they do for this notebook's own kernel above. (An older `sitecustomize.py` / `PYTHONPATH` workaround was needed only before `retrieval_caps.py` was wired directly into `app.py`; it is no longer part of the current implementation.) `GEMMA_USE_4BIT=0` selects full bf16, which is 3-5x faster than 4-bit (NF4) given the VRAM headroom freed by the previous step.


In [8]:
import subprocess, time, os, requests

# Values match the current repository's Dockerfile ENV defaults exactly
# (RAG_MAX_CANDIDATES=60, RAG_MAX_CONTEXT=12, GEMMA_MAX_NEW_TOKENS=1536,
# GEMMA_USE_4BIT=0) -- see the Docker Support section below. No
# PYTHONPATH/sitecustomize.py is needed: src/app.py imports
# src/retrieval_caps.py directly at module load time.
server_env = {
    **os.environ,
    "GEMMA_USE_4BIT": "0",
    "GEMMA_MAX_NEW_TOKENS": "1536",
    "RAG_MAX_CANDIDATES": "60",   # read by src/retrieval_caps.py
    "RAG_MAX_CONTEXT": "12",
}

server = subprocess.Popen(
    ["uvicorn", "src.app:app", "--host", "127.0.0.1", "--port", "8000"],
    env=server_env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

print("Starting server (bf16, production-matched caps active)...")
start = time.time()
while True:
    if server.poll() is not None:
        print("Server exited early:")
        print(server.stdout.read())
        break
    try:
        r = requests.get("http://127.0.0.1:8000/status", timeout=3)
        if r.status_code == 200:
            print(f"HTTP up after {time.time()-start:.0f}s -> {r.json()}")
            break
    except requests.exceptions.RequestException:
        pass
    print(f"  ...{time.time()-start:.0f}s")
    time.sleep(5)


Starting server (bf16, production-matched caps active)...
  ...0s
  ...5s
  ...10s
  ...15s
  ...20s
  ...25s
  ...30s
  ...35s
HTTP up after 40s -> {'status': 'running', 'embedding_model': 'BAAI/bge-m3', 'reranker_model': 'BAAI/bge-reranker-v2-m3', 'dense_model_loaded': True, 'reranker_model_loaded': True, 'gemma_model_loaded': True, 'chromadb_connected': True}


**Expected Output**
A polling loop against `/status` until the server reports healthy, ending with the parsed status JSON: `{'status': 'running', 'embedding_model': 'BAAI/bge-m3', 'reranker_model': 'BAAI/bge-reranker-v2-m3', 'dense_model_loaded': True, 'reranker_model_loaded': True, 'gemma_model_loaded': True, 'chromadb_connected': True}`.

**Conclusion**
The FastAPI application is up, fully initialized, and ready to serve `/ask` requests with the same retrieval caps and bf16 model configuration validated earlier in this notebook -- now matching the repository's actual Docker production defaults exactly, rather than a narrower notebook-only override.


## 13. API Testing

**Purpose**
Confirm the `/ask` endpoint responds correctly end to end before timing real questions.

**Explanation**
The server process's first `/ask` call includes the full in-process model load, so this warm-up call is run once and excluded from the latency figures gathered next.


In [9]:
print("Warming up (first call includes the full model load)...")
t0 = time.time()
try:
    w = requests.post("http://127.0.0.1:8000/ask", json={"query": "warmup"}, timeout=1800)
    print(f"Warm-up done in {time.time()-t0:.0f}s -- status {w.status_code}")
except requests.exceptions.RequestException as e:
    print(f"Warm-up failed after {time.time()-t0:.0f}s: {e}")
print("\nModel resident. Timings below are pure inference.")


Warming up (first call includes the full model load)...
Warm-up done in 1s -- status 200

Model resident. Timings below are pure inference.


**Expected Output**
HTTP `200` status with the warm-up elapsed time. Warm-up is expected to be fast if the server's models were already resident from the `/status` polling loop above (no cold load), and to take materially longer (a full Gemma-2-9B cold load) if this is the very first request the server process has handled.

**Conclusion**
The API is verified live and responsive; timings from here on reflect real query latency, not initialization overhead.


## 14. Docker Support

**Purpose**
Document how this validated pipeline is containerized for production, per the current
`dmrc_deploy` repository's `Dockerfile`.

**Explanation**
No code is run in this notebook cell — this documents the real `Dockerfile` committed to
`dmrc_deploy`, which packages the exact FastAPI application (`src/app.py`) this notebook
validates against a live `uvicorn` process:

- Base image: `nvidia/cuda:12.1.1-runtime-ubuntu22.04`, with Python 3.11 installed on top.
- Dependencies install with the same `bitsandbytes`/`nvidia-nvjitlink-cu13` best-effort
  pattern used in this notebook's own **Loading Gemma-2-9B-it** section — so a missing 4-bit
  wheel never fails the whole image build.
- `src/`, `data/`, and the pre-built `chroma_db/` are copied directly into the image — no
  ingestion or rebuild step runs at container start.
- `retrieval_caps.py` is a version-controlled module imported directly by `app.py`, so unlike
  an older pre-audit deployment approach, the container needs no `sitecustomize.py` /
  `PYTHONPATH` trick at all — `app.py` is the container's own entrypoint and gets the caps
  automatically on import, exactly as this notebook's **FastAPI Integration** section above
  now does too.
- `HEALTHCHECK` polls `GET /status` with a **25-minute** `start-period`, since a cold start on
  an empty model volume downloads ~2.3GB (BGE-M3) + ~2.3GB (reranker) + ~18GB (Gemma-2-9B
  bf16) before `/status` can report healthy.
- Container env defaults: `RAG_MAX_CANDIDATES=60`, `RAG_MAX_CONTEXT=12`, `GEMMA_USE_4BIT=0`,
  `GEMMA_MAX_NEW_TOKENS=1536` — the exact same values the **FastAPI Integration** section
  above now launches the notebook's own server process with.

**Observation**
The Dockerfile's own inline comments record a forensic-audit history: earlier
`RAG_MAX_CANDIDATES`/`RAG_MAX_CONTEXT` values (20/8) were found to silently re-truncate
clauses that a later widening of `app.py`'s own retrieval constants (`MERGED_CANDIDATE_POOL`,
`RERANK_TOP_N`) had already fixed — the Dockerfile has since been updated to stay in sync with
those constants rather than re-imposing the old ceiling.

**Conclusion**
This notebook validates the application logic (model load, capped retrieval, FastAPI
serving) that the container image runs unchanged via the same `uvicorn src.app:app`
entrypoint — Docker build/run itself is outside Colab's capabilities and is exercised on the
GPU host (RunPod) at deploy time instead.


## 15. Production Configuration

**Purpose**
Consolidate the environment-driven configuration surface this pipeline exposes, per the
current `dmrc_deploy` repository.

**Explanation**
No new code is run here. Every value below is read directly from an environment variable at
process start, with the repository's own code-level default shown alongside the value this
notebook's own server-launch cell (and the Dockerfile) actually use in production:

| Variable | Read by | Code-level default | This notebook / Dockerfile |
|---|---|---|---|
| `RAG_MAX_CANDIDATES` | `retrieval_caps.py` | `30` | `60` |
| `RAG_MAX_CONTEXT` | `retrieval_caps.py` | `15` | `12` |
| `GEMMA_MAX_NEW_TOKENS` | `gemma_inference.py` | `1536` | `1536` |
| `GEMMA_USE_4BIT` | `gemma_inference.py` | `0` (off) | `0` |
| `RAG_MIN_CONFIDENCE` | `reranker.py` (`evaluate_confidence`) | `0.10` | `0.10` |
| `RAG_MIN_SEPARATION` | `reranker.py` (`evaluate_confidence`) | `0.08` | `0.08` |
| `ALLOWED_ORIGINS` | `app.py` (CORS) | `"*"` (local-dev only) | set explicitly at deploy time |
| `RAG_DEBUG` | `app.py`, `hybrid_retriever.py`, `reranker.py`, `prompt_engineering.py` | `0` (off) | `0` |

**Observation**
`RAG_MAX_CANDIDATES`/`RAG_MAX_CONTEXT` are deliberately wider in the Docker image (60/12) than
the bare code-level default (30/15) -- both are safely above `app.py`'s own
`MERGED_CANDIDATE_POOL=60`/`RERANK_TOP_N=12` retrieval constants, so neither ceiling
re-truncates output the retrieval layer already tuned. `RAG_MIN_CONFIDENCE`/`RAG_MIN_SEPARATION`
gate the low-confidence short-circuit in `app.py`'s `/ask` endpoint (see **Final End-to-End
Demonstration** below) and are explicitly flagged in `reranker.py` as needing recalibration
against real query logs before being fully trusted in production.

**Conclusion**
Production tuning is entirely configuration-driven. This notebook's own server-launch cell
above uses the exact same values the committed Dockerfile does, so behaviour validated in this
Colab session is representative of the deployed container.


## 16. Error Handling

**Purpose**
Document the failure-handling behaviour exercised on both the client side (this notebook)
and the server side (the current `dmrc_deploy` repository's `/ask` implementation).

**Explanation**
No new code is run here. On the **client side**, drawn directly from **API Testing** and
**Final End-to-End Demonstration** above: every call to `/ask` is wrapped in
`try/except requests.exceptions.RequestException`, HTTP status codes other than `200` are
surfaced with the response body rather than silently ignored, and every attempt is timed and
recorded (success or failure) so a failing query never silently drops out of the results
summary.

On the **server side**, `src/app.py` wraps each pipeline stage — hybrid retrieval, reranking,
prompt construction, and Gemma inference — in its own `try/except`, converting any failure
into an explicit `HTTPException(status_code=500, ...)` with a stage-specific detail message,
rather than letting an unhandled exception in one stage produce an opaque server error.
Recoverable failures are handled without failing the request at all: sibling expansion falls
back to the reranker's own output on failure, and the model warm-up steps in the server's
`lifespan` handler log and continue rather than blocking startup if a model fails to warm up
(it is then loaded lazily on the first real request instead).

**Conclusion**
Both this notebook's demonstration script and the production API it calls follow the same
defensive pattern — timeout-bounded, explicitly-checked, and failure-tolerant rather than
silently swallowing or crashing on error.


## 17. Performance Notes

**Purpose**
Summarize what this notebook's timing cells measure, and how to read the figures once you re-run them.

**Explanation**
This notebook times four independent things, each in the cell that produces it -- re-run this
notebook end to end to get current figures for your own session (a Colab GPU allocation, model
download/cache state, and free-text query length all affect these numbers, so no single
recorded figure stays representative across sessions or configuration changes):

| Stage | Where it's measured | What changed this configuration |
|---|---|---|
| Raw generation (bf16) | **Model Loading**, `generate_answer()`'s own `tok/s` print | Unchanged -- still `google/gemma-2-9b-it`, `attn_implementation="eager"` |
| Capped retrieval + rerank + prompt build (in-kernel) | **Building Final Prompt** | `MAX_CONTEXT` raised from the old `4` to the current repository default `15`; more grounding context now reaches the prompt at a correspondingly larger (still in-budget) size |
| Server startup to healthy `/status` | **FastAPI Integration** | Server now launches with `RAG_MAX_CANDIDATES=60`/`RAG_MAX_CONTEXT=12`/`GEMMA_MAX_NEW_TOKENS=1536` (Docker-matched), not the old `20`/`4`/`320` |
| `/ask` end-to-end, 6 real queries (5 free-text + 1 BOQ) | **Final End-to-End Demonstration** | Query set now includes a BOQ-grounded question to exercise the BOQ document-name fallback |

**Conclusion**
With retrieval caps, exact token-budget enforcement, and bf16 generation all active, this
pipeline is designed to keep real `/ask` queries well inside the context window and out of the
multi-minute-timeout territory previously diagnosed as caused by uncapped retrieval and
unintended CPU offload -- re-run the cells above in your own Colab session to confirm current
timings.


## 18. Final End-to-End Demonstration

**Purpose**
Exercise the complete, capped, bf16-served pipeline against real free-text and BOQ contract
queries, in one batch, as the final validation of this notebook.

**Explanation**
Each query goes through the full live stack: exact-clause-number fast path check → hybrid
retrieval (dense + BM25) → confidence gate → capped rerank (+ sibling expansion) → capped/
token-budgeted prompt build → Gemma 2 9B generation, over HTTP, exactly as a real client of the
deployed API would call it. The demonstration script below prints only the answer text and the
leading source's retrieval path, for brevity — the full `/ask` response schema (per the current
`dmrc_deploy` repository's `src/app.py`) carries considerably more:

- **Source citations** — every entry in the response's `sources` array carries `clause`,
  `page`, `document`, `item_number`, `retrieval_source`, `reranker_score`, and `chunk_id`,
  resolved from the reranked candidate's own metadata. For a BOQ-sourced answer, `item_number`
  is resolved from the row's `s_no` field (not `item_number`, which only exists on clause
  metadata as a cross-reference) via `prompt_engineering.get_boq_item_number()`.
- **BOQ document-name fallback** — `prompt_engineering.get_document_name()` returns
  `document_title` or `document_name` when present, and otherwise falls back to a BOQ row's
  `source_pdf` field (cleaned up into a readable title), since BOQ metadata never carries
  `document_title`/`document_name` the way clause metadata does. Both `app.py`'s `sources`
  list and the prompt text Gemma sees resolve this identically, via the same shared helper.
- **Confidence score** — the response's top-level `confidence` field reports the strongest
  `reranker_score` actually backing the returned answer (`None` when no candidates were used).
  A separate distribution-based check (`evaluate_confidence`) runs on the reranked candidates
  *before* Gemma is called: if the top reranker score falls below a configurable floor
  (`RAG_MIN_CONFIDENCE`, default `0.10`), or isn't meaningfully separated from the rest of the
  candidate pool (`RAG_MIN_SEPARATION`, default `0.08`), the endpoint returns a fixed
  "not found in context" answer with an empty `sources` list and skips generation entirely,
  rather than letting Gemma answer from weak or off-topic context.

Neither of these behaviours is exercised in isolation by this notebook's own executed cells --
they are documented here as the current production response contract, and the BOQ query added
to the batch below is the notebook's own concrete exercise of the document-name fallback.


In [10]:
QUERIES = [
    "What are the contractor obligations?",
    "Who is responsible for maintenance during the defects liability period?",
    "What training must the contractor provide to employer staff?",
    "What are the requirements for the environmental control system software?",
    "Which stations are covered under this contract?",
    "What is the quantity and rate for the cooling tower BOQ item?",  # BOQ-grounded query --
                                                                        # exercises the BOQ
                                                                        # document-name fallback
                                                                        # (source_pdf) and s_no
                                                                        # item-number mapping.
]

results = []
for q in QUERIES:
    t0 = time.time()
    try:
        r = requests.post("http://127.0.0.1:8000/ask", json={"query": q}, timeout=300)
        dt = time.time() - t0
        if r.status_code == 200:
            data = r.json()
            src = data.get("sources", [])
            how = src[0].get("retrieval_source") if src else "-"
            conf = data.get("confidence")
            print(f"\n{'='*74}\nQ: {q}\n   [{dt:.1f}s | {len(src)} sources | path: {how} | confidence: {conf}]")
            print(f"\n{data.get('answer','')}")
            results.append((q, dt, True))
        else:
            print(f"\nQ: {q}\n   HTTP {r.status_code}: {r.text[:200]}")
            results.append((q, dt, False))
    except requests.exceptions.RequestException as e:
        dt = time.time() - t0
        print(f"\nQ: {q}\n   FAILED after {dt:.0f}s: {type(e).__name__}")
        results.append((q, dt, False))

print(f"\n\n{'='*74}\nSUMMARY")
for q, dt, ok in results:
    print(f"  {'OK ' if ok else 'FAIL'} {dt:6.1f}s  {q[:56]}")
ok_times = [d for _, d, o in results if o]
if ok_times:
    print(f"\n  {len(ok_times)}/{len(results)} succeeded | avg {sum(ok_times)/len(ok_times):.1f}s")



Q: What are the contractor obligations?
   [75.3s | 12 sources | path: dense+sparse | confidence: 0.809]

The Contractor has several obligations as detailed in the provided documents:

* **Interfacing with Authorities:** The Contractor is responsible for obtaining any necessary "clearances or certificates" from local authorities to ensure the ECS system is fully functional and operational as per the Contract. (Clause 4.2, Page 4)
* **Submission Management:** The Contractor must maintain a complete and organized file of all submissions, including an index and locating system to track the status of each submission. This index must be available for the Employer's Representative's review and used to assign sequential numbers to deliverables and track resubmissions. Supplemental information must accompany each submission to fully explain the equipment and its intended use. (Clause 6.10.3, Page 8)
* **Spare Testing:** The Contractor must ensure all spares are correctly calibrated, tested, a

**Expected Output**
For each query: elapsed time, source count, retrieval path, confidence score, and the generated
answer; followed by a summary table and success rate. The five free-text queries are expected
to resolve via `dense+sparse` (hybrid) retrieval, grounded in `clause`-type chunks with
`Clause N, Page P` citations. The BOQ query is expected to resolve to one or more `boq`-type
chunks, with citations built from `s_no` (not `clause_no`) and a document name resolved via the
`source_pdf` fallback described above (e.g. a title derived from a filename like
`Contract_Agreement_..._Vol-3-28-37.pdf` rather than a clause document title).

**Conclusion**
The full RAG pipeline — retrieval, hybrid fusion, reranking, capped/token-budgeted prompt
construction, and grounded Gemma 2 9B generation — is validated end to end over live HTTP
across both contract-clause and Bill-of-Quantities content, producing accurate, cited,
in-budget answers with an explicit confidence score on every response.


### Shut down the server

Run this last, after all `/ask` queries above are done, to release the server process and its GPU memory.


In [11]:
server.terminate()
try:
    server.wait(timeout=30)
    print("Server stopped.")
except subprocess.TimeoutExpired:
    server.kill()
    server.wait(timeout=10)
    print("Server didn't exit gracefully in 30s; force-killed it.")


Server stopped.


**Expected Output** `Server stopped.` (or a forced-kill message if it didn't exit within 30s.) **Observation** The server process terminated cleanly. **Conclusion** GPU memory is released back to the runtime; the session can be safely ended or reused.


## 19. Conclusion

This notebook validated the complete generation and serving stage of the DMRC Contract Intelligence RAG pipeline, against the current `dmrc_deploy` repository:

- **Model** — `google/gemma-2-9b-it` authenticated, loaded in bf16 on a GPU runtime, and smoke-tested for correct generation before any retrieval-grounded query is attempted.
- **Quantization** — a configuration-driven 4-bit path (`GEMMA_USE_4BIT=1`) exists for smaller GPUs; this session validated the full-bf16 path (`GEMMA_USE_4BIT=0`, the repository default).
- **Retrieval caps** — `src/retrieval_caps.py`, a permanent committed module imported directly by `app.py`, bounds `hybrid_search()`, `rerank()`, `expand_with_siblings()`, and `get_chunks_by_parent_clause()` to `RAG_MAX_CANDIDATES=30`/`RAG_MAX_CONTEXT=15` by code default (`60`/`12` in the Docker image), protecting against unbounded prompt growth on free-text queries.
- **Token budget enforcement / Gemma context-window protection** — `prompt_engineering.fit_context_to_budget()` tokenizes the assembled prompt with the exact tokenizer `generate_answer()` uses, reserves `GEMMA_MAX_NEW_TOKENS` (1536 by default) plus a measured chat-template overhead, and trims the lowest-ranked surviving chunks first until the prompt provably fits inside `gemma_inference.MODEL_CONTEXT_WINDOW_TOKENS` (8192 tokens) — an exact guarantee, not a character-count estimate.
- **BOQ document fallback** — `prompt_engineering.get_document_name()` resolves a readable document title for BOQ rows from their `source_pdf` metadata field when no `document_title`/`document_name` is present, and `get_boq_item_number()`/`get_boq_page_number()` resolve BOQ-specific citation fields (`s_no`, `page_number`) — used identically by both the LLM prompt and the API's `sources` response.
- **Confidence score** — `reranker.evaluate_confidence()` gates generation on the reranked candidate pool's own score distribution (absolute floor `RAG_MIN_CONFIDENCE=0.10`, separation margin `RAG_MIN_SEPARATION=0.08`); the API's `/ask` response reports the strongest `reranker_score` backing each answer as its top-level `confidence` field.
- **Serving** — the same pipeline was served over a live FastAPI process (`uvicorn src.app:app`), reaching healthy status with every model component (embedding, reranker, Gemma) confirmed loaded via `GET /status`.
- **End-to-end demonstration** — six real questions (five free-text contract questions plus one BOQ-grounded question) answered and cited via the live `/ask` endpoint.

Combined with notebook 01's retrieval validation (which now also covers the corpus's 290 BOQ chunks alongside its 63 clause chunks), this constitutes a full, reproducible validation of the DMRC Contract Intelligence RAG backend — from a persisted ChromaDB collection through hybrid retrieval, reranking, capped and token-budgeted generation, to a live, production-configured API — against this repository's current implementation only.
